In [1]:
import tensorflow as tf
from tensorflow.contrib import layers
from tensorflow.contrib import rnn
import datetime
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
from LCII_hyper_parameters_preference import hyper_parameters
from utils_Integrated import IIRNNDataHandler

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 生成走勢圖
def png(epoch, value_1, value_2, value_3, value_4, value_5):
    
    fig = plt.figure(figsize=(6,4))
    fig.patch.set_facecolor('white')
    plt.rcParams['font.sans-serif'] = ['Yu Gothic']
    plt.grid(True)
    
    if epoch == "loss_epoch":
        plt.plot(df_3[epoch], df_3[value_1])
        plt.plot(df_3[epoch], df_3[value_2])
        plt.legend([value_1, value_2], loc="upper left")
        
    elif epoch == "train_epoch":
        plt.plot(df_1[epoch], df_1[value_1])
        plt.plot(df_1[epoch], df_1[value_2])
        plt.plot(df_1[epoch], df_1[value_3])
        plt.legend([value_1, value_2, value_3], loc="upper left")
        
    elif epoch == "test_epoch":  
        plt.plot(df_2[epoch], df_2[value_1])
        plt.plot(df_2[epoch], df_2[value_2])
        plt.plot(df_2[epoch], df_2[value_3])
        plt.legend([value_1, value_2, value_3], loc="upper left")
    
    plt.xlabel("Epoch")    
    if epoch == "loss_epoch":
        plt.ylabel("loss")
        
    else:
        plt.ylabel("%")
    plt.title(value_5)
    plt.savefig("./testlog/"+ value_4 + ".png")
    #plt.clf()

In [ ]:
#整合策略計算
def integrated_computing (switch_plot, input_1, input_2):
    if switch_plot == "sum":
        input_sum = input_1 + input_2

    elif switch_plot == "dot":
        input_sum = input_1 * input_2

    elif switch_plot == "attention_gate_sum":
        a = tf.math.tanh(input_1)
        b = tf.math.tanh(input_2)
        a = tf.nn.softmax(a)
        b = tf.nn.softmax(b)
        ab_sum = a+b
        input_sum = (a/ab_sum*input_1) + (b/ab_sum*input_2)

    elif switch_plot == "attention_gate_dot":
        w_input_1 = tf.math.tanh(input_1)
        w_input_2 = tf.math.tanh(input_2)
        w_input_1 = tf.nn.softmax(w_input_1)
        w_input_2 = tf.nn.softmax(w_input_2)
        w_input_sum = w_input_1+w_input_2
        input_sum = ((w_input_1/w_input_sum)*input_1) * ((w_input_2/w_input_sum)*input_2)

    return input_sum

In [ ]:
model_num = hyper_parameters.model_num

for run in range(model_num):
    print("==========================================\n")
    code_runtime_now = datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S')
    runtime = time.time()
    tf.reset_default_graph()
    print("LCII Program Running Progress: "+ str(run+1)+"/"+str(model_num)+" | Current Time: "+str(code_runtime_now))
    dataset = hyper_parameters.run_list[run][0] #reddit / lastfm / instacart / tmall / amazon / MovieLens_1M / MovieLens_20M / steam
    switch_plot = hyper_parameters.run_list[run][1] #sum / dot / attention_gate_sum / attention_gate_dot
    switch_initial_state = hyper_parameters.run_list[run][2] #True / False
    fusion_way = hyper_parameters.run_list[run][3] #att / lp / fix / none
    strategy = hyper_parameters.run_list[run][4] #pre-combine / post-combine / original
    windows = hyper_parameters.run_list[run][5] #0-100% #意旨短期只抓最鄰近的X比例之偏好作短期，其餘則都當長期
    long_score = hyper_parameters.run_list[run][6] #有值填數值0.0-1.0；反之'no_use'
    short_score = hyper_parameters.run_list[run][7] #有值填數值0.0-1.0；反之'no_use'
    EMBEDDING_SIZE = INNER_RNN_SIZE = OUTER_RNN_SIZE = hyper_parameters.run_list[run][8] #Embedding Size
    BATCHSIZE = hyper_parameters.run_list[run][9]  #一次讀取多少筆資料
    learning_rate = hyper_parameters.run_list[run][10]
    dropout_pkeep = hyper_parameters.run_list[run][11]
    MAX_EPOCHS = hyper_parameters.run_list[run][12]
    MAX_SESSION_REPRESENTATIONS = 0 #不會用到
    log_file_name = hyper_parameters.run_list[run][13] #在副程式會自動命名
    use_FC = hyper_parameters.run_list[run][15] #是否要在input_sum 加入FC #True / False
    
    do_training = True #False
    save_best = True

    isExists_file = os.path.exists('./testlog')
    if not isExists_file:
        os.makedirs('./testlog') 
        
        
    dataset_path = './Datasets/'+dataset+'/4_train_test_split.pickle'
    #dataset_path = './Datasets/'+dataset+'/4_train_test_split_sapmle.pickle'
    epoch_file = './epoch_file-'+log_file_name+'-'+dataset+'.pickle'
    checkpoint_file = './checkpoints/'+log_file_name+'-'+dataset+'-'
    checkpoint_file_ending = '.ckpt'
    date_now = datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d')
    log_file = './Testlog/'+ str(date_now) + '-testing-' + 'Integrated_log_preference_1' + '.txt' #區別開來(_1/_2) RTX2080只能同時跑2個程式

    
    reddit = "subreddit"
    lastfm = "lastfm"
    instacart = "instacart"
    tmall = "tmall_6"
    amazon = "amazon"
    MovieLens_1M = 'MovieLens-1M'
    MovieLens_20M = 'MovieLens-20M'
    steam= 'steam'
    
    if dataset == reddit:
        from test_util_ver_final_ver_len20 import Tester
        SEQLEN = 20-1

    elif dataset == lastfm:
        from test_util_ver_final_ver_len20 import Tester
        SEQLEN = 20-1

    elif dataset == instacart:
        from test_util_ver_final_ver_len20 import Tester
        SEQLEN = 20-1

    elif dataset == tmall:
        from test_util_ver_final_ver_len20 import Tester
        SEQLEN = 20-1

    elif dataset == amazon:
        from test_util_ver_final_ver_len20 import Tester
        SEQLEN = 20-1

    elif dataset == MovieLens_1M:
        from test_util_ver_final_ver_len200 import Tester
        SEQLEN = 200-1
        
    elif dataset == MovieLens_20M:
        from test_util_ver_final_ver_len20 import Tester

    elif dataset == steam:
        from test_util_ver_final_ver_len15 import Tester
        SEQLEN = 15-1
        
    seed = 0
    tf.set_random_seed(seed)
    N_ITEMS      = -1
    N_LAYERS     = 1        # number of layers in the rnn
    TOP_K = 20

    # Load training data
    datahandler = IIRNNDataHandler(dataset_path, BATCHSIZE, log_file, 
            MAX_SESSION_REPRESENTATIONS, INNER_RNN_SIZE)
    N_ITEMS = datahandler.get_num_items() 
    N_SESSIONS = datahandler.get_num_training_sessions()
    
    message = "------------------------------------------\n"
    message += "DATASET: "+dataset+" | MODEL: "+ log_file_name
    message += "\nSWITCH_INITIAL_STATE: "+ str(switch_initial_state)+" | USE_FC_IN_INPUT-SUM: "+ str(use_FC)
    message += "\nSTRATEGY: "+strategy+" | FUSION_WAY: "+fusion_way+" | SWITCH_PLOT: "+switch_plot
    message += "\nADJACENT_WINDOW_RATIO(0-100%): "+str(windows)+" | LONG_TERM_SCORE: "+str(long_score)+" | SHORT_TERM_SCORE: "+str(short_score)
    message += "\nDROPOUT: "+str(dropout_pkeep)+" | LEARNING_RATE: "+str(learning_rate)+" | BATCHSIZE: "+str(BATCHSIZE)+" | SEED: "+str(seed)
    message += "\nOUTER_RNN_SIZE: "+str(OUTER_RNN_SIZE)+" | INNER_RNN_SIZE: "+str(INNER_RNN_SIZE)+" | EMBEDDING_SIZE: "+str(EMBEDDING_SIZE)
    message += "\nTRAIN_N_SESSIONS: "+str(N_SESSIONS)+" | N_ITEMS: "+str(N_ITEMS)+" | N_LAYERS: "+str(N_LAYERS)+" | SEQLEN: "+str(SEQLEN)+" | MAX_EPOCHS: "+str(MAX_EPOCHS)
    #message += "\nMAX_SESSION_REPRESENTATIONS: "+str(MAX_SESSION_REPRESENTATIONS)
    message += "\n------------------------------------------\n"
    datahandler.log_config(message)

    if not do_training:
        print("\nOBS!!!! Training is turned off!\n")
    ##
    ## The model
    ##
    print("Creating model")

    # Inputs
    X = tf.placeholder(tf.int32, [None, None], name='X')    # [ BATCHSIZE, SEQLEN ]
    Y_ = tf.placeholder(tf.int32, [None, None], name='Y_')  # [ BATCHSIZE, SEQLEN ]

    # Embeddings. W_embed = all embeddings. X_embed = retrieved embeddings 
    # from W_embed, corresponding to the items in the current batch
    W_embed = tf.Variable(tf.random_uniform([N_ITEMS, EMBEDDING_SIZE], -1.0, 1.0), name='embeddings')
    X_embed = tf.nn.embedding_lookup(W_embed, X) # [BATCHSIZE, SEQLEN, EMBEDDING_SIZE]
    
    # Length of sesssions (not considering padding)
    seq_len = tf.placeholder(tf.int32, [None], name='seqlen')
    batchsize = tf.placeholder(tf.int32, name='batchsize')

    lr = tf.placeholder(tf.float32, name='lr')              # learning rate
    pkeep = tf.placeholder(tf.float32, name='pkeep')        # dropout parameter

    # Inner_RNN
    Inner_cell = rnn.GRUCell(INNER_RNN_SIZE)
    Inner_dropcell = rnn.DropoutWrapper(Inner_cell, input_keep_prob=pkeep, output_keep_prob=pkeep)
    Inner_outputs, Inner_states = tf.nn.dynamic_rnn(Inner_dropcell, X_embed,
            sequence_length=seq_len, dtype=tf.float32)
        
    if strategy == 'original':
        input_sum = integrated_computing(switch_plot, X_embed, Inner_outputs)
        
    else:
        #長短期處理
        #h_list.append(tf.nn.relu(Inner_outputs))#Relu
        if int((SEQLEN+1) * windows * 0.01) < 1:
            capture_len_h_list = -1
        else:
            capture_len_h_list = int((SEQLEN+1) * -windows * 0.01)
           
        short_sum = Inner_outputs[:, capture_len_h_list:]
        long_sum  = Inner_outputs[:, :capture_len_h_list]   
        

        if fusion_way == "lp":
            #學習長短期權重
            W_embed_Lp_long = tf.Variable(tf.random_uniform([1, SEQLEN+capture_len_h_list, OUTER_RNN_SIZE], -1.0, 1.0), name='embeddings_Lp_long')
            W_embed_Lp_short = tf.Variable(tf.random_uniform([1, -capture_len_h_list, OUTER_RNN_SIZE], -1.0, 1.0), name='embeddings_Lp_short')
            sess_rep = tf.concat([(long_sum*W_embed_Lp_long), (short_sum*W_embed_Lp_short)], 1) #long先再來Short
            
        elif fusion_way == "fix":
            sess_rep = tf.concat([(long_sum*long_score), (short_sum*short_score)], 1)
            
        elif fusion_way == "att":
            short_x = tf.gather_nd(short_sum, tf.stack([tf.range(batchsize), seq_len-1], axis=1)) 
            long_x = tf.gather_nd(long_sum, tf.stack([tf.range(batchsize), seq_len-1], axis=1)) 
            
            short_x = tf.math.tanh(short_x)
            long_x  = tf.math.tanh(long_x)
            short_x = tf.nn.softmax(short_x)
            long_x  = tf.nn.softmax(long_x)
            short_long_sum = short_x+long_x
            
            max_short = short_x/short_long_sum
            max_long  = long_x/short_long_sum
            
            max_short_2 = tf.argmax((short_x/short_long_sum), 1)
            max_long_2  = tf.argmax((long_x/short_long_sum), 1)
            max_short_2 = tf.cast(max_short_2[0],dtype=tf.float32)
            max_long_2 = tf.cast(max_long_2[0],dtype=tf.float32)
            sess_rep = tf.concat([(long_sum*max_long_2), (short_sum*max_short_2)], 1)
            
        See_long_short_term = sess_rep
        
        if strategy == 'pre-combine':
            input_sum = integrated_computing(switch_plot, X_embed, sess_rep)            
            
        elif strategy == 'post-combine':
            input_sum = integrated_computing(switch_plot, X_embed, Inner_outputs)
            input_sum = input_sum * sess_rep
            
            
    if use_FC == True:
        input_sum = layers.linear(input_sum, EMBEDDING_SIZE) #是否最後在input_sum 加入FC
    
    last_Inner_output = tf.gather_nd(Inner_outputs, tf.stack([tf.range(batchsize), seq_len-1], axis=1)) 

    # Outer_RNN
    onecell = rnn.GRUCell(OUTER_RNN_SIZE)
    dropcell = rnn.DropoutWrapper(onecell, input_keep_prob=pkeep)
    multicell = rnn.MultiRNNCell([dropcell]*N_LAYERS, state_is_tuple=False)
    multicell = rnn.DropoutWrapper(multicell, output_keep_prob=pkeep)

    if switch_initial_state == True:
        Yr, H = tf.nn.dynamic_rnn(multicell, input_sum, sequence_length=seq_len, dtype=tf.float32, 
                                  initial_state=last_Inner_output)
    else:
        Yr, H = tf.nn.dynamic_rnn(multicell, input_sum, sequence_length=seq_len, dtype=tf.float32)

    H = tf.identity(H, name='H') # just to give it a name

    # Apply softmax to the output
    # Flatten the RNN output first, to share weights across the unrolled time steps
    Yflat = tf.reshape(Yr, [-1, OUTER_RNN_SIZE])         # [ BATCHSIZE x SEQLEN, OUTER_RNN_SIZE ]
    # Change from internal size (from RNNCell) to N_ITEMS size
    Ylogits = layers.linear(Yflat, N_ITEMS)                     # [ BATCHSIZE x SEQLEN, N_ITEMS ]

    # Flatten expected outputs to match actual outputs
    Y_flat_target = tf.reshape(Y_, [-1])    # [ BATCHSIZE x SEQLEN ]

    # Calculate loss
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(logits=Ylogits, labels=Y_flat_target)    # [ BATCHSIZE x SEQLEN ]

    # Mask the losses (so we don't train in padded values)
    mask = tf.sign(tf.to_float(Y_flat_target))
    masked_loss = mask * loss

    # Unflatten loss
    loss = tf.reshape(masked_loss, [batchsize, -1])            # [ BATCHSIZE, SEQLEN ]

    # Get the index of the highest scoring prediction through Y
    Y = tf.argmax(Ylogits, 1)   # [ BATCHSIZE x SEQLEN ]
    Y = tf.reshape(Y, [batchsize, -1], name='Y')        # [ BATCHSIZE, SEQLEN ]

    # Get prediction
    top_k_values, top_k_predictions = tf.nn.top_k(Ylogits, k=TOP_K)        # [BATCHSIZE x SEQLEN, TOP_K]
    Y_prediction = tf.reshape(top_k_predictions, [batchsize, -1, TOP_K], name='YTopKPred')

    # Training
    train_step = tf.train.AdamOptimizer(lr).minimize(loss)

    # Stats
    # Average sequence loss
    seqloss = tf.reduce_mean(loss, 1)
    # Average batchloss
    batchloss = tf.reduce_mean(seqloss)

    # Average number of correct predictions
    accuracy = tf.reduce_mean(tf.cast(tf.equal(Y_, tf.cast(Y, tf.int32)), tf.float32))
    loss_summary = tf.summary.scalar("batch_loss", batchloss)
    acc_summary = tf.summary.scalar("batch_accuracy", accuracy)
    summaries = tf.summary.merge([loss_summary, acc_summary])

    # Init to save models
    if not os.path.exists("checkpoints"):
        os.mkdir("checkpoints")
    saver = tf.train.Saver(max_to_keep=1)

    # Initialization
    init = tf.global_variables_initializer()
    config = tf.ConfigProto(allow_soft_placement=True)
    config.gpu_options.allow_growth = True      # be nice and don't use more memory than necessary
    sess = tf.Session(config=config)
    saver = tf.train.Saver()

    ##
    ##  TRAINING
    ##
    print(message) #顯示本次實驗狀態
    print()
    print("Starting training.")
    print()
    epoch = datahandler.get_latest_epoch(epoch_file)
    print("|-Starting on epoch", epoch+1)
    if epoch > 0:
        print("|--Restoring model.")
        save_file = checkpoint_file + checkpoint_file_ending
        saver.restore(sess, save_file)
    else:
        sess.run(init)
    epoch += 1


    best_recall5 = -1
    best_recall20 = -1

    num_training_batches = datahandler.get_num_training_batches()
    num_test_batches = datahandler.get_num_test_batches()

    # 生成走勢圖
    data = [[], [], [], [], [], [], [], [], []]

    temp_loss = 1000000000000000 #防止第一次就判定loss上升
    Threshold = hyper_parameters.run_list[run][14] #recall@5如果超過 x 就判定overfitting

    while epoch <= MAX_EPOCHS:
        print()
        print("Starting epoch #"+str(epoch))
        epoch_loss = 0
        # 生成走勢圖
        data[0].append(epoch) #epoch

        datahandler.reset_user_batch_data()
        datahandler.reset_user_session_representations()

        if do_training:
            tester = Tester()
            _batch_number = 0
            xinput, targetvalues, sl, session_reps, sr_sl, user_list = datahandler.get_next_train_batch()

            while len(xinput) > int(BATCHSIZE/2):
                _batch_number += 1
                batch_start_time = time.time()

                feed_dict = {X: xinput, Y_: targetvalues, lr: learning_rate, pkeep: dropout_pkeep, 
                        batchsize: len(xinput), seq_len: sl}
                
                batch_predictions, _, bl, sess_rep = sess.run([Y_prediction, train_step, batchloss, H], feed_dict=feed_dict)
                #see_max_short, see_max_long, see_max_short_2, see_max_long_2, see_long_short_term, see_short_sum, see_long_sum, batch_predictions, _, bl, sess_rep = sess.run([max_short, max_long, max_short_2, max_long_2, See_long_short_term, short_sum, long_sum, Y_prediction, train_step, batchloss, H], feed_dict=feed_dict)

                # Evaluate predictions
                tester.evaluate_batch(batch_predictions, targetvalues, sl)

                batch_runtime = time.time() - batch_start_time
                epoch_loss += bl
                
                if _batch_number == 1:
                    print("Train | Batch number:", str(_batch_number), "/", str(num_training_batches))
                    #print("Train | Batch number:", str(_batch_number), "/", str(num_training_batches), "| Long & Short-term R:%s | Short-term R:%s | Long-term R:%s" %(see_long_short_term.shape, see_short_sum.shape, see_long_sum.shape))
                if _batch_number%100 == 0:
                    print("Train | Batch number:", str(_batch_number), "/", str(num_training_batches), "| Batch time:", "%.2f" % batch_runtime, " seconds", end='')
                    print(" | Batch loss:", bl, end='')
                    eta = (batch_runtime*(num_training_batches-_batch_number))/60
                    eta = "%.2f" % eta
                    print(" | ETA:", eta, "minutes.")

                xinput, targetvalues, sl, session_reps, sr_sl, user_list = datahandler.get_next_train_batch()
            
            #print("Train | Batch number:", str(_batch_number), "/", str(num_training_batches), "| Long & Short-term R:%s | Short-term R:%s | Long-term R:%s" %(see_long_short_term.shape, see_short_sum.shape, see_long_sum.shape))
            #print("Train | see_max_short:%s | see_max_long:%s | see_max_short_2:%s | see_max_long_2:%s" %(see_max_short.shape, see_max_long.shape, see_max_short_2, see_max_long_2))
            print("Epoch", epoch, "finished")
            print("|- Train Loss:", epoch_loss)
            # 生成走勢圖
            data[7].append(epoch_loss) #epoch_loss_train
            if epoch_loss > temp_loss:
                print("The loss increases, and the best solution in the whole domain may be found !")
                print()
            temp_loss = epoch_loss

            # Print final test stats for epoch
            test_stats, current_recall5, current_recall10, current_recall20, current_mrr5, current_mrr10, current_mrr20, current_ndcg5, current_ndcg10, current_ndcg20 = tester.get_stats_and_reset()
            temp_recall5 = current_recall5
            print("Recall@5    : "  +str(current_recall5)    +" | "+"Recall@10    : "  +str(current_recall10)    +" | "+"Recall@20    : "  +str(current_recall20))
            print("MRR@5       : "  +str(current_mrr5)       +" | "+"MRR@10       : "  +str(current_mrr10)       +" | "+"MRR@20       : "  +str(current_mrr20))
            print("NDCG@5      : "  +str(current_ndcg5)      +" | "+"NDCG@10      : "  +str(current_ndcg10)      +" | "+"NDCG@20      : "  +str(current_ndcg20))
            print("--------------------------------------------------------------------------------------")
            # 生成走勢圖
            data[1].append(current_recall20) #Recall@20
            data[2].append(current_mrr20) #MRR@20
            data[3].append(current_ndcg20) #NDCG@20


        ##
        ##  TESTING
        ##
        print("Starting testing")
        tester = Tester()
        datahandler.reset_user_batch_data()
        _batch_number = 0
        epoch_loss_test = 0
        xinput, targetvalues, sl, session_reps, sr_sl, user_list = datahandler.get_next_test_batch()

        while len(xinput) > int(BATCHSIZE/2):
            batch_start_time = time.time()
            _batch_number += 1

            feed_dict = {X: xinput, Y_: targetvalues, pkeep: 1.0, batchsize: len(xinput), seq_len: sl}

            bl_t, batch_predictions, sess_rep = sess.run([batchloss, Y_prediction, H], feed_dict=feed_dict) #1:預測項目(y) 2:偏好表示(h)

            # Evaluate predictions
            tester.evaluate_batch(batch_predictions, targetvalues, sl)

            # Print some stats during testing
            batch_runtime = time.time() - batch_start_time
            epoch_loss_test += bl_t
            
            
            if _batch_number%100 == 0:
                print("Test | Batch number:", str(_batch_number), "/", str(num_test_batches), "| Batch time:", "%.2f" % batch_runtime, " seconds", end='')
                eta = (batch_runtime*(num_test_batches-_batch_number))/60
                eta = "%.2f" % eta
                print(" ETA:", eta, "minutes.")

            xinput, targetvalues, sl, session_reps, sr_sl, user_list = datahandler.get_next_test_batch()

   
        print("|- Test Loss:", epoch_loss_test)
        # 生成走勢圖
        data[8].append(epoch_loss_test) #epoch_loss_test
        # Print final test stats for epoch
        test_stats, current_recall5, current_recall10, current_recall20, current_mrr5, current_mrr10, current_mrr20, current_ndcg5, current_ndcg10, current_ndcg20 = tester.get_stats_and_reset()
        print("Recall@5    : "  +str(current_recall5)    +" | "+"Recall@10    : "  +str(current_recall10)    +" | "+"Recall@20    : "  +str(current_recall20))
        print("MRR@5       : "  +str(current_mrr5)       +" | "+"MRR@10       : "  +str(current_mrr10)       +" | "+"MRR@20       : "  +str(current_mrr20))
        print("NDCG@5      : "  +str(current_ndcg5)      +" | "+"NDCG@10      : "  +str(current_ndcg10)      +" | "+"NDCG@20      : "  +str(current_ndcg20))
        # 生成走勢圖
        data[4].append(current_recall20) #Recall@20
        data[5].append(current_mrr20) #MRR@20
        data[6].append(current_ndcg20) #NDCG@20

        lgr = temp_recall5*100
        if Threshold < lgr and epoch > 1: #recall@5大於Threshold 且epoch > 1
            print()
            print("<<<<<<<<<<<<<<<<<<<<<<< OVERFITTING ! >>>>>>>>>>>>>>>>>>>>>>>")
            print()

        if save_best:
            if current_recall5 > best_recall5 and Threshold > lgr: #如果recall@5比過去來的好，且recall@5小於Threshold%
                # Save the model
                print("Saving model.")
                save_file = checkpoint_file + checkpoint_file_ending
                save_path = saver.save(sess, save_file)
                print("|- Model saved in file:", save_path)

                best_recall5 = current_recall5

                #save best once
                temp_list = []
                temp_list.append([epoch, current_recall5, current_recall10, current_recall20, current_mrr5, current_mrr10, current_mrr20, current_ndcg5, current_ndcg10, current_ndcg20, epoch_loss, test_stats])

                datahandler.store_current_epoch(epoch, epoch_file)
                #datahandler.log_test_stats(epoch, epoch_loss, test_stats)

        epoch += 1

    print()
    print("BEST Epoch |", temp_list[0][0])
    print("Recall@5    : "  +str(temp_list[0][1])    +" | "+"Recall@10    : "  +str(temp_list[0][2])    +" | "+"Recall@20    : "  +str(temp_list[0][3]))
    print("MRR@5       : "  +str(temp_list[0][4])    +" | "+"MRR@10       : "  +str(temp_list[0][5])    +" | "+"MRR@20       : "  +str(temp_list[0][6]))
    print("NDCG@5      : "  +str(temp_list[0][7])    +" | "+"NDCG@10      : "  +str(temp_list[0][8])    +" | "+"NDCG@20      : "  +str(temp_list[0][9]))
    print()
    datahandler.log_test_stats(temp_list[0][0], temp_list[0][10], temp_list[0][11]) #只節錄最佳的成績寫入log
    
    end_time_1 = (time.time() - runtime)/60
    end_time_2 = (time.time() - runtime)/60/60
    end_time_1 = "%.2f" % end_time_1
    end_time_2 = "%.2f" % end_time_2
    print("Runtime Spend:", str(end_time_1), "min | "+str(end_time_2), "hr")
    print()
    
    # 生成走勢圖
    df_1 = pd.DataFrame({"train_epoch": data[0], "Recall@20": data[1], "MRR@20": data[2], "NDCG@20": data[3]}) #訓練
    df_2 = pd.DataFrame({"test_epoch": data[0], "Recall@20": data[4], "MRR@20": data[5], "NDCG@20": data[6]}) #測試
    df_3 = pd.DataFrame({"loss_epoch": data[0], "train_loss": data[7], "test_loss": data[8]})
    png("train_epoch", "Recall@20", "MRR@20", "NDCG@20", dataset+"-"+log_file_name+"-k@20評估指標走勢圖_train", "k@20評估指標走勢圖_train")#訓練
    png("test_epoch", "Recall@20", "MRR@20", "NDCG@20", dataset+"-"+log_file_name+"-k@20評估指標走勢圖_test", "k@20評估指標走勢圖_test")  #測試
    png("loss_epoch", "train_loss", "test_loss", "none", dataset+"-"+log_file_name+"-loss走勢圖", "loss走勢圖")
    
    datahandler.close_log() #終止輸入log